# Week 1 — LLM Fundamentals & Environment Setup

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-01-llm-fundamentals-content.html`. Run the cells in order during lab time.

**You will practice:**
1. Making your first Gemini API call from Python.
2. Seeing `temperature`, `top_p`, and `top_k` change model output on the *same* prompt.
3. Counting tokens for real and comparing it to the "1 token ≈ 4 characters" rule of thumb.
4. Making the *same* call through three tools: the raw SDK, a minimal Google ADK agent, and a minimal LangChain chat model.
5. Two open exercises.

**Before you start:** create a free API key at [Google AI Studio](https://aistudio.google.com/app/apikey) and save it in a `.env` file next to this notebook:

```
GOOGLE_API_KEY="paste-your-key-here"
```


In [2]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Environment check

Load the API key and make one call to confirm everything is wired up.

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in a .env file first."

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"

response = client.models.generate_content(
    model=MODEL,
    contents="response. yes ",
)
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


A large language model is an artificial intelligence system trained on vast amounts of text data to understand, process, and generate human-like language.


## 2. Sampling parameters: temperature, top-p, top-k

We'll send the **same prompt** several times, changing only `temperature`. Watch how the output goes from
predictable to varied.

In [9]:
PROMPT = "Give me one creative name for a coffee shop. Reply with just the name."

for temp in [0.0, 0.4, 0.9, 1.5]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=temp, max_output_tokens=500),
    )
    print(f"temperature={temp:<4} -> {response.text}")
    

temperature=0.0  -> Morning Alchemy
temperature=0.4  -> Sonder & Steam
temperature=0.9  -> Fable & Froth
temperature=1.5  -> Perch & Pour


Run the cell above a few times. At `temperature=0.0` the answer should barely change between runs.
At `temperature=1.5` you should see real variety (and occasionally something a little unhinged — that's expected).

Now let's isolate `top_p` and `top_k` by holding temperature fixed at a mid-range value.

In [10]:
for top_p in [0.1, 0.5, 0.95]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_p=top_p, max_output_tokens=500),
    )
    print(f"top_p={top_p:<5} -> {response.text}")

print()
for top_k in [1, 5, 40]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_k=top_k, max_output_tokens=500),
    )
    print(f"top_k={top_k:<3} -> {response.text}")

top_p=0.1   -> Fable & Foam
top_p=0.5   -> Roast & Reverie
top_p=0.95  -> Paper & Pour

top_k=1   -> Fable & Foam
top_k=5   -> Fable & Froth
top_k=40  -> Paper & Pour


## 3. Tokens: count them for real

The rule of thumb is "1 token ≈ 4 characters ≈ 0.75 words" for English. Let's check it against the real
tokenizer using `count_tokens`.

In [ ]:
samples = [
    "Hi!",
    "The capital of France is Paris.",
    "AI Agentic Engineering is a course about building autonomous systems with large language models, "
    "retrieval-augmented generation, and multi-agent orchestration frameworks like Google ADK and LangGraph.",
    "Una vez que hayas terminado de leer este texto, por favor, responde con un resumen en español. No olvides la puntuación y las reglas gramaticales.",
]

for text in samples:
    result = client.models.count_tokens(model=MODEL, contents=text)
    chars = len(text)
    tokens = result.total_tokens
    ratio = chars / tokens if tokens else 0
    print(f"chars={chars:<4} tokens={tokens:<4} chars/token={ratio:.2f}  | {text[:50]!r}")

chars=3    tokens=3    chars/token=1.00  | 'Hi!'
chars=31   tokens=8    chars/token=3.88  | 'The capital of France is Paris.'
chars=200  tokens=36   chars/token=5.56  | 'AI Agentic Engineering is a course about building '
chars=146  tokens=33   chars/token=4.42  | 'Una vez que hayas terminado de leer este texto, po'


## 4. Same call, three tools

Below, the identical prompt is sent through: **(A)** the raw `google-genai` SDK, **(B)** a one-line **Google ADK**
agent run with `InMemoryRunner`, and **(C)** a **LangChain** chat model. This is a *light preview* — we are not
building real agents yet (tools, multi-step reasoning, ReAct) until Corte 2, Week 6 onward. The goal today is
just to recognize the same underlying API call under each interface.

In [15]:
# --- A. Raw SDK ---
prompt = "Name one famous physicist and the discovery they're best known for."

response = client.models.generate_content(model=MODEL, contents=prompt)
print("A) raw SDK:\n", response.text)

A) raw SDK:
 **Albert Einstein**, who is best known for developing the **Theory of General Relativity**, which fundamentally changed our understanding of gravity, space, and time.


In [16]:
# --- B. Google ADK (minimal agent) ---
import asyncio
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

adk_agent = Agent(
    model=MODEL,
    name="week1_demo_agent",
    instruction="Answer concisely, in 1-2 sentences.",
)

async def ask_adk_agent(agent, prompt, app_name="week1_app", user_id="student"):
    """Small reusable helper: runs one prompt through an ADK agent and returns the final text."""
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = await runner.session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    for event in runner.run(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

print("B) Google ADK agent:\n", await ask_adk_agent(adk_agent, prompt))

B) Google ADK agent:
 Albert Einstein is best known for developing the theory of relativity, which fundamentally revolutionized our understanding of space, time, and gravity.


In [17]:
# --- C. LangChain chat model ---
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=MODEL)
lc_response = llm.invoke(prompt)
print("C) LangChain:\n", lc_response.content)

C) LangChain:
 [{'type': 'text', 'text': '**Albert Einstein** is best known for developing the **Theory of General Relativity**, which revolutionized our understanding of gravity, space, and time.', 'extras': {'signature': 'ErMMCrAMARFNMg8rrfCa0y7x3oU+MFWY4YeHRU8YVjvvdBbXZr/KmH2KHPSaDWEpooo6fVSOE4N80QiuDxIrRq4ptbYKvcc33/xtsxiEXoQ9bUV8onUDFZ5LR4IRA1XOjC8rU/sM3jthumqFZu58kGOYgHLx0HI5kzp9pwg/To5dvk1XfDYuA5Bo87Wyc3eXO1xZ9RYpnsR/uNsbEboX/Cl7D2O1OUuO78rRWBYAYALwq1NVfRMjZ8f0Y1jMA+r3jDxYE+GWGRYH6+At4jhc3L1X39M7jNr7vpovjdTChbHEAnNg7BtwkV3/DP/ivVIbe78+oP2jpDBBUfL5VCNrj/LmFYKV7r7h5E6GGGYI/J1gJYqD6WMPPsnOq7zyDhwd7qXKt2t9z5SfHucAPWIoIDtS8PVuC8BxRB9xMm99PY60li+PqCf80IqVpihSJkII7PjU+IARA/Yvdz0l77ypw8HxiJTvHztU6i2IlrAVN2qgV4cumlbCukJYKbSbC7AZ2p2espCryw1PjgmoEuoTHe/izKxnKE2V2UzZOPsP2cSSPCOI5t+Hp39yiWK+5CEmsZVCB0Mp3r4INQIVYlMqHiWBBHUI5w9TBUE7CYHR5cjjdjilgav7muvqO7RgmQiRwas7+loPE1pIx63uIDPep2G7+BLlJIQuO/aF88wAu9odfhsZxfW8nnH7A8RZmvlwv3VXXFt4+T4r88/ABGt42cLA3fHhfakjr4rcRLWt0Yfe0wNR3r7d39SATxjuYOuzaXqLljte

## 5. Exercises

Complete both before the peer code review activity.

In [ ]:
# Exercise 1 — Your own parameter sweep
# Factual framing: only one correct-ish answer, low variety expected even at high temperature.

MY_PROMPT = "Explain in one sentence why the sun is hot"

for temp in [0.0, 0.3, 0.7, 1.2, 1.8]:
    response = client.models.generate_content(
        model=MODEL,
        contents=MY_PROMPT,
        config=types.GenerateContentConfig(temperature=temp, max_output_tokens=200),
    )
    print(f"temperature={temp:<4} -> {response.text}")

SyntaxError: f-string: expecting '}' (3208691943.py, line 12)

In [ ]:
# Exercise 2 — Estimate tokens without calling the API
# Write a function `estimate_tokens(text)` that approximates token count using ONLY the
# "1 token ≈ 4 characters" rule (no API call). Then compare it against the real count_tokens()
# result for the three `samples` from section 3, and print the % error for each.

def estimate_tokens(text: str) -> float:
    return len(text) / 4

for text in samples:
    real = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    estimate = estimate_tokens(text)
    error_pct = abs(estimate - real) / real * 100 if real else 0
    print(f"real={real:<4} estimate={estimate:<6.1f} error={error_pct:5.1f}%  | {text[:50]!r}")

## 6. Activity 3b — Prompt & Parameter Detective (solo version)

The in-class version has the instructor hand each group three outputs from undisclosed `temperature`/`top_p` settings. Doing it solo with your local model: generate the three outputs yourself, but shuffle and hide the labels so you're analyzing them blind, same as in class.

Run the next cell, then **before printing** `_answer_key`:

1. Rank `Output A/B/C` from most to least deterministic-looking.
2. Justify each ranking in one sentence using: sampling, distribution, nucleus, peaked/flat.
3. Say which setting you'd ship for: (a) a support-ticket categorizer, (b) a creative story generator, (c) a JSON-producing data-extraction endpoint.
4. Only then reveal `_answer_key` and check how close your ranking was.

In [4]:
import random

PROMPT_DETECTIVE = "Write a one-sentence opening for a daily story."

SETTINGS = [
    {"label": "low",  "temperature": 0.0, "top_p": 0.1},
    {"label": "mid",  "temperature": 0.7, "top_p": 0.9},
    {"label": "high", "temperature": 1.4, "top_p": 1.0},
]

shuffled = SETTINGS.copy()
random.shuffle(shuffled)

outputs = []
for cfg in shuffled:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT_DETECTIVE,
        config=types.GenerateContentConfig(
            temperature=cfg["temperature"], top_p=cfg["top_p"], max_output_tokens=500
        ),
    )
    outputs.append(response.text.strip())

for i, text in enumerate(outputs):
    print(f"Output {chr(65+i)}:\n{text}\n")

_answer_key = {chr(65 + i): shuffled[i] for i in range(len(shuffled))}

Output A:
:* "By the time the steam had cleared from the bathroom mirror, Arthur had already decided to live

Output B:
The morning smelled of rain and burnt toast, a familiar comfort that made the sudden silence from the street

Output C:
The world always felt full of quiet possibility in the ten minutes between the



In [5]:
# Reveal — run this only after you've written your ranking and justification above.

print("True settings, from most to least deterministic:")
for label in sorted(_answer_key, key=lambda l: (_answer_key[l]["temperature"], _answer_key[l]["top_p"])):
    cfg = _answer_key[label]
    print(f"  Output {label}: temperature={cfg['temperature']}, top_p={cfg['top_p']}  ({cfg['label']})")

True settings, from most to least deterministic:
  Output C: temperature=0.0, top_p=0.1  (low)
  Output A: temperature=0.7, top_p=0.9  (mid)
  Output B: temperature=1.4, top_p=1.0  (high)


## Next week

Week 2 — **Context Engineering I**: system prompts, few-shot prompting, chain-of-thought, and forcing structured
JSON output. See `week-02-context-engineering-i-content.html`.